# 📗 Kit 2 — Modélisation & Pipeline ETL
## Formation Data Engineer

---

### Comment utiliser ce notebook ?

- **`# ✅ CODE FOURNI`** → lisez attentivement et exécutez
- **`# 🔧 TODO`** → vous devez compléter ce bloc

### Prérequis
- Avoir `dataset_final_kit1.csv` dans le même dossier

### Livrables attendus
- `memo_formes_normales.md`
- `data/clean/communes.csv`, `secteurs.csv`, `entreprises.csv`
- `pipeline.py` + `pipeline.log`
- `entreprises_isere.db`


In [2]:
# ✅ CODE FOURNI — Imports
import pandas as pd
import sqlite3
import logging
import json
import os
import shutil

os.makedirs('data/clean', exist_ok=True)
os.makedirs('data/raw',   exist_ok=True)

if os.path.exists('dataset_final_kit2.csv') and not os.path.exists('data/raw/dataset_final_kit2.csv'):
    shutil.copy('dataset_final_kit2.csv', 'data/raw/dataset_final_kit2.csv')

SRC = 'data/raw/dataset_final_kit2.csv'

if not os.path.exists(SRC):
    print(f'❌ {SRC} introuvable — terminez le Kit 2 d\'abord')
else:
    df = pd.read_csv(SRC)
    print(f'✓ Dataset chargé : {len(df)} lignes, {len(df.columns)} colonnes')
    print(f'Colonnes : {df.columns.tolist()}')


✓ Dataset chargé : 490 lignes, 11 colonnes
Colonnes : ['nom_commune', 'nom', 'siren', 'code_naf', 'tranche_effectif', 'adresse', 'cp_entreprise', 'latitude', 'longitude', 'libelle_naf', 'libelle_tranche']


---
## Section 4 — Partie 4 : Analyse & Modélisation UML

### Objectif
Analyser le dataset avec Python pour identifier les redondances et déterminer quelles entités doivent être séparées dans votre diagramme UML.

### Qu'est-ce qu'une redondance ?
La latitude et longitude de Grenoble sont copiées dans TOUTES les lignes des entreprises de Grenoble. Dans un bon modèle, ces informations ne sont stockées qu'**une seule fois** dans une table Commune.


In [3]:
# 🔧 TODO — Exercice 4.1 : Analyser la cardinalité
#
# 1. Affichez le nombre de valeurs uniques par colonne
# 2. Quelles colonnes ont beaucoup de répétitions ?

pd.options.display.max_columns = None
print('=== 5 premières lignes ===')
print(df.head(10).to_string())

print('\n=== Cardinalité (valeurs uniques par colonne) ===')
for col in df.columns:
    print(f'  {col:25s} : {df[col].nunique()}')  # TODO : df[col].nunique()
    

=== 5 premières lignes ===
  nom_commune                        nom      siren code_naf tranche_effectif                                                                  adresse  cp_entreprise  latitude  longitude                                                             libelle_naf          libelle_tranche
0    GRENOBLE                   LA POSTE  356000000   53.10Z               53  DIRECTION GENERALE DE LA POSTE 9 RUE DU COLONEL PIERRE AVIA 75015 PARIS          75015   45.1842     5.7155  Activités de poste dans le cadre d'une obligation de service universel  10 000 salariés et plus
1    GRENOBLE      ELECTRICITE DE FRANCE  552081317   35.11Z               53                                    22-30 22 AVENUE DE WAGRAM 75008 PARIS          75008   45.1842     5.7155                                                Production d'électricité  10 000 salariés et plus
2    GRENOBLE  ELIOR RESTAURATION FRANCE  662025196   56.29B               53                                       51 CH

In [4]:
# ✅ CODE FOURNI — Analyse des redondances
# Objectif : identifier quelles colonnes se répètent inutilement
# et en déduire quelles entités gagneraient à être séparées dans le modèle.

print('=== ANALYSE DES REDONDANCES ===')

# Les coords GPS sont-elles constantes pour une commune ?
# Si oui → elles appartiennent à Commune, pas à Entreprise
check = df.groupby('nom_commune')[['latitude','longitude']].nunique()
print(f'Communes avec coords incohérentes : {len(check[check.max(axis=1) > 1])}')
print('  → lat/lon se répètent par commune mais restent identiques')
print('  → candidat : table Commune séparée')

# Un SIREN identifie-t-il une seule entreprise ?
siren_multi = df.groupby('siren')['nom_commune'].nunique()
print(f'\nSIREN dans plusieurs communes : {(siren_multi > 1).sum()}')
print('  → siren identifie une entreprise unique : candidat PK de Entreprise')

# Le code_naf est-il partagé entre plusieurs entreprises ?
print(f'\nNombre de code_naf distincts : {df["code_naf"].nunique()}')
print(f'Entreprises par code_naf en moyenne : {len(df) / max(df["code_naf"].nunique(),1):.1f}')
print('  → code_naf partagé entre entreprises : candidat à une table Secteur séparée')

print('\n=== OBSERVATIONS ===')
print('3 groupes de données redondantes identifiés :')
print('  1. Informations géographiques de la commune (lat, lon, code_postal...)')
print('  2. Informations sur le secteur d\'activité (code_naf, libelle_naf...)')
print('  3. Informations sur la tranche d\'effectif (code_tranche, libelle_tranche...)')
print()
print('→ Ces observations vont guider votre diagramme UML.')
print('  Les formes normales (Section 5) formaliseront ces intuitions.')
print()
print('📌 Créez maintenant votre diagramme UML v1 sur draw.io — sauvegarder en diagramme_uml.png')


=== ANALYSE DES REDONDANCES ===
Communes avec coords incohérentes : 0
  → lat/lon se répètent par commune mais restent identiques
  → candidat : table Commune séparée

SIREN dans plusieurs communes : 24
  → siren identifie une entreprise unique : candidat PK de Entreprise

Nombre de code_naf distincts : 25
Entreprises par code_naf en moyenne : 19.6
  → code_naf partagé entre entreprises : candidat à une table Secteur séparée

=== OBSERVATIONS ===
3 groupes de données redondantes identifiés :
  1. Informations géographiques de la commune (lat, lon, code_postal...)
  2. Informations sur le secteur d'activité (code_naf, libelle_naf...)
  3. Informations sur la tranche d'effectif (code_tranche, libelle_tranche...)

→ Ces observations vont guider votre diagramme UML.
  Les formes normales (Section 5) formaliseront ces intuitions.

📌 Créez maintenant votre diagramme UML v1 sur draw.io — sauvegarder en diagramme_uml.png


In [5]:
# 🔧 TODO — Q2, Q3 & Q4 : Préparer le diagramme UML
#
# Q2. Si le code postal de Grenoble change, combien de lignes
#     faut-il modifier dans le dataset plat ?
#     Répondez en commentaire.
#
# Q3. Identifiez au moins 3 entités distinctes dans le dataset.
#     Listez leurs attributs en commentaire.
#
# Q4. Créez le diagramme UML v1 sur draw.io ou Excalidraw.
#     Sauvegardez en diagramme_uml.png.

# TODO Q2 :
# Dataset plat    : ???
#######il faudrait modifier toutes les lignes de la commune concernée, avec un risque d'incohérence si on en oublie une. 
# Table Commune   : ???
####### Dans un modèle avec table communes séparée : une seule ligne à modifier, 
####### et toutes les entreprises de cette commune en bénéficient automatiquement via la clé étrangère.


# TODO Q3 — entités et attributs :
# commune (code_insee PK, nom_commune)
# entreprise (siren PK, nom, adresse, cp_entreprise, tranche_effectif, + FK vers Commune et Secteur)
# secteur (code_naf PK, libelle_naf)

# TODO Q4 : diagramme à créer sur draw.io — sauvegarder en diagramme_uml.png
print('Q4 : Créez votre diagramme UML v1 sur draw.io ou Excalidraw')


Q4 : Créez votre diagramme UML v1 sur draw.io ou Excalidraw


---
## Section 5 — Partie 5 : Formes normales & Scripts de transformation

### Rappel des formes normales

**1NF** : chaque cellule = une valeur atomique (pas de listes). Chaque ligne est unique.

**2NF** : en 1NF + tous les attributs dépendent de la TOTALITÉ de la clé primaire. Ne s'applique qu'aux clés composites. Si PK = siren (une colonne), 1NF implique automatiquement 2NF.

**3NF** : en 2NF + pas de dépendance transitive. `siren → code_naf → libelle_naf` est une violation. Solution : table Secteur séparée.


In [15]:
# 🔧 TODO — Exercice 5.1 : Rédiger le mémo des formes normales
#
# Complétez les ??? et sauvegardez dans memo_formes_normales.md

memo_lines = [
    '# Mémo : Formes Normales\n',
    '## Appliquées au dataset entreprises Isère\n',
    '\n',
    '## 1NF\n',
    'Définition : chaque cellule contient une seule valeur atomique (pas de listes, pas de valeurs séparées par des virgules). Chaque ligne est unique.\n',
    'Exemple de violation dans notre dataset : chaque cellule = une seule valeur. Dans notre dataset, adresse contient parfois rue + ville + CP collés ensemble → violation. Correction : séparer en colonnes distinctes.\n',
    'Notre dataset est-il en 1NF ? non\n',
    '\n',
    '## 2NF\n',
    'Définition : en 1NF + tous les attributs dépendent de la TOTALITÉ de la clé primaire. Ne s applique qu aux clés composites. Si PK = siren (une colonne), 1NF implique automatiquement 2NF.\n',
    'À quoi s\'applique-t-elle ? pas de dépendance partielle sur une clé composite. Dans notre dataset, si on avait une PK (siren, nom_commune), le nom de l entreprise ne dépend que du siren → violation. Solution : table Entreprise avec PK = siren seul.\n',
    'Notre dataset est-il en 2NF ? Oui\n',
    '\n',
    '## 3NF\n',
    'Définition : en 2NF + pas de dépendance transitive. `siren → code_naf → libelle_naf` est une violation. Solution : table Secteur séparée.\n',
    'Violation dans notre dataset : Le libelle_naf dépend du code_naf, pas du siren\n',
    'Solution appliquée : table Secteur(code_naf PK, libelle_naf)\n',
    '\n',
    '## Conclusion\n',
    'À quelle forme normale correspond le dataset plat ? Le diagramme v1 est en 2NF : les entités ont des PK simples et pas de dépendance partielle. \n',
    'À quelle forme normale correspond notre modèle cible (3 tables) ? 3NF\n',
]

with open('memo_formes_normales.md', 'w', encoding='utf-8') as f:
    f.writelines(memo_lines)
print('✓ memo_formes_normales.md créé (à compléter !)')


✓ memo_formes_normales.md créé (à compléter !)


### Exercice 5.2 — Scripts de transformation

Voici `extract_communes()` fourni comme modèle. Lisez attentivement avant d'écrire les deux autres.


In [63]:
# ✅ CODE FOURNI — Q8 : extract_communes() — modèle de référence
# Lisez chaque commentaire attentivement avant d'écrire les deux suivants.

def extract_communes(src, output='data/clean/communes.csv'):
    df = pd.read_csv(src)
    # drop_duplicates : une ligne par commune (le dataset plat a N lignes par commune)
    c = df[['nom_commune','latitude','longitude']]\
         .drop_duplicates('nom_commune').copy()
    c.columns = ['nom_commune','latitude','longitude']
    # Code INSEE simulé : 38 (Isère) + numéro séquentiel
    c.insert(0, 'code_insee', [f'38{i:03d}' for i in range(1, len(c)+1)])
    c.to_csv(output, index=False)  # index=False : pas de colonne numérique inutile
    print(f'extract_communes : {len(c)} communes → {output}')
    return c

df_communes = extract_communes(SRC)
print(df_communes.head(3).to_string())


extract_communes : 49 communes → data/clean/communes.csv
   code_insee nom_commune  latitude  longitude
0       38001    GRENOBLE   45.1842     5.7155
10      38002      VOIRON   45.3805     5.5869
20      38003      VIENNE   45.5221     4.8803


In [45]:
# 🔧 TODO — Q9 : extract_secteurs() et extract_tranches()
#
# Sur le modèle de extract_communes(), écrivez extract_secteurs() qui :
# - extrait les colonnes code_naf ET libelle_naf (les deux sont maintenant dans le dataset)
# - dédoublonne sur code_naf
# - supprime les lignes sans code_naf
# - sauvegarde dans data/clean/secteurs.csv
#
# Puis écrivez extract_tranches() sur le même modèle :
# - extrait code_tranche ET libelle_tranche
# - dédoublonne sur code_tranche
# - sauvegarde dans data/clean/tranches_effectifs.csv



def extract_secteurs(src, output='data/clean/secteurs.csv'):
    df = pd.read_csv(src)
    
    s = df[['code_naf', 'libelle_naf']] \
        .dropna(subset=['code_naf']).drop_duplicates('code_naf').copy()
    
    s.to_csv(output, index=False)
    print(f'extract_secteurs : {len(s)} → {output}')
    return s


def extract_tranches(src, output='data/clean/tranches_effectifs.csv'):
    df = pd.read_csv(src)
    
    t = df[['tranche_effectif', 'libelle_tranche']].drop_duplicates('tranche_effectif').copy()
    
    t.to_csv(output, index=False)
    print(f'extract_tranches : {len(t)} → {output}')
    return t

df_secteurs  = extract_secteurs(SRC)
df_tranches  = extract_tranches(SRC)
print(df_secteurs.head())
print(df_tranches.head())


extract_secteurs : 25 → data/clean/secteurs.csv
extract_tranches : 8 → data/clean/tranches_effectifs.csv
  code_naf                                        libelle_naf
0   53.10Z  Activités de poste dans le cadre d'une obligat...
1   35.11Z                           Production d'électricité
2   56.29B             Autres services de restauration n.c.a.
3   64.19Z                  Autres intermédiations monétaires
4   52.21Z     Services auxiliaires des transports terrestres
    tranche_effectif          libelle_tranche
0                 53  10 000 salariés et plus
7                 11         10 à 19 salariés
79                52   5 000 à 9 999 salariés
117               NN   Unités non employeuses
155               02           3 à 5 salariés


In [47]:
# 🔧 TODO — Q10 & Q11 : extract_entreprises()
#
# Q10. Écrivez extract_entreprises() :
#      - normalise nom_commune des deux côtés avant le merge
#      - merge LEFT pour obtenir le code_insee
#      - sélectionne : siren, nom, code_naf, code_insee, code_tranche, adresse
#        (on garde code_tranche comme FK vers tranches_effectifs)
#      - dédoublonne sur siren, supprime les lignes sans siren
#
# Q11. Testez le script et vérifiez les résultats avant de continuer.

def extract_entreprises(src, communes_path, output='data/clean/entreprises.csv'):
    df  = pd.read_csv(src)
    com = pd.read_csv(communes_path)

    com['nom_clean'] = com['nom_commune'].str.lower().str.strip()
    df['nom_clean']  = df['nom_commune'].str.lower().str.strip()

    #  Merge LEFT pour récupérer code_insee
    df_merge = df.merge(com[['nom_clean', 'code_insee']], on='nom_clean', how='left')

    taux = df_merge['code_insee'].isnull().mean() * 100
    print(f'  Entreprises sans code_insee : {taux:.1f}%')

    # code_tranche (FK vers tranches_effectifs) remplace tranche_effectif
    e = df_merge[['siren', 'nom', 'code_naf', 'code_insee', 'tranche_effectif', 'adresse']].copy()
    e = e.rename(columns={'tranche_effectif': 'code_tranche'})
    e = e.drop_duplicates(subset=['siren'])
    e = e.dropna(subset=['siren'])


    e.to_csv(output, index=False)
    print(f'extract_entreprises : {len(e)} → {output}')
    return e

df_entreprises = extract_entreprises(SRC, 'data/clean/communes.csv')
print(df_entreprises.head(3).to_string())




  Entreprises sans code_insee : 0.0%
extract_entreprises : 36 → data/clean/entreprises.csv
       siren                        nom code_naf  code_insee code_tranche                                                                  adresse
0  356000000                   LA POSTE   53.10Z       38001           53  DIRECTION GENERALE DE LA POSTE 9 RUE DU COLONEL PIERRE AVIA 75015 PARIS
1  552081317      ELECTRICITE DE FRANCE   35.11Z       38001           53                                    22-30 22 AVENUE DE WAGRAM 75008 PARIS
2  662025196  ELIOR RESTAURATION FRANCE   56.29B       38001           53                                       51 CHEMIN DES MECHES 94000 CRETEIL


---
## Section 6 — Partie 6 : Pipeline ETL & SQLite

### Le pipeline ETL
1. **Extract** : lire les données brutes
2. **Transform** : appeler nos fonctions extract_*
3. **Load** : charger dans SQLite

### SQLite : base de données dans un fichier
SQLite stocke tout dans un fichier `.db`. Python intègre sqlite3 en standard.

**Les 3 lignes obligatoires :**
```python
conn = sqlite3.connect('ma_base.db')  # Crée ou ouvre
conn.commit()                          # OBLIGATOIRE : persiste les changements
conn.close()                           # Ferme la connexion
```

⚠️ Sans `conn.commit()`, les insertions sont en mémoire et perdues !


In [59]:
# ✅ CODE FOURNI — load_sqlite() avec table de rejets et tranches_effectifs

DDL = '''
DROP TABLE IF EXISTS rejected_rows;
DROP TABLE IF EXISTS entreprises;
DROP TABLE IF EXISTS tranches_effectifs;
DROP TABLE IF EXISTS communes;
DROP TABLE IF EXISTS secteurs;

CREATE TABLE communes(
    code_insee  TEXT PRIMARY KEY,
    nom_commune         TEXT NOT NULL,
    latitude    REAL,
    longitude   REAL
);

CREATE TABLE secteurs(
    code_naf    TEXT PRIMARY KEY,
    libelle_naf TEXT
);

CREATE TABLE tranches_effectifs(
    tranche_effectif    TEXT PRIMARY KEY,
    libelle_tranche TEXT
);

CREATE TABLE entreprises(
    siren            TEXT PRIMARY KEY,
    nom              TEXT,
    code_naf         TEXT REFERENCES secteurs(code_naf),
    code_insee       TEXT REFERENCES communes(code_insee),
    code_tranche     TEXT REFERENCES tranches_effectifs(code_tranche),
    adresse          TEXT
);

CREATE TABLE rejected_rows(
    id         INTEGER PRIMARY KEY AUTOINCREMENT,
    table_name TEXT,
    row_data   TEXT,
    error_msg  TEXT,
    ts         TEXT DEFAULT (datetime('now'))
);
'''

def safe_insert(cursor, table, df):
    ok = err = 0
    for _, row in df.iterrows():
        d = {k: (None if pd.isna(v) else v) for k, v in row.items()}
        try:
            cols = ','.join(d.keys())
            ph   = ','.join(['?'] * len(d))
            cursor.execute(f'INSERT OR IGNORE INTO {table}({cols}) VALUES({ph})', list(d.values()))
            ok += 1
        except Exception as e:
            cursor.execute(
                'INSERT INTO rejected_rows(table_name,row_data,error_msg) VALUES(?,?,?)',
                [table, str(d), str(e)]
            )
            err += 1
    print(f'  {table:25s} : {ok:5d} insérées, {err:3d} rejetées')

def load_to_sqlite(db_path, df_communes, df_secteurs, df_tranches, df_entreprises):
    conn = sqlite3.connect(db_path)
    conn.executescript(DDL)
    cur  = conn.cursor()
    # Ordre important : charger les tables référencées avant entreprises
    safe_insert(cur, 'communes',          df_communes)
    safe_insert(cur, 'secteurs',          df_secteurs)
    safe_insert(cur, 'tranches_effectifs', df_tranches)
    safe_insert(cur, 'entreprises',       df_entreprises)
    conn.commit()
    conn.close()
    print(f'✓ Base SQLite créée : {db_path}')

print('✓ Fonctions définies')


✓ Fonctions définies


In [64]:
# 🔧 TODO — Q12, Q13 & Q14 : Logique du pipeline
#
# Q12. Quelle approche avons-nous utilisée (ETL ou ELT) ? Justifiez en 2 phrases.
#
# Q13. Écrivez run_pipeline() qui orchestre les 3 étapes avec logging.
#      Attention : load_to_sqlite attend maintenant 5 arguments
#      (db, communes, secteurs, tranches, entreprises).
#
# Q14. load_to_sqlite est fourni. Expliquez en commentaire :
#      - pourquoi OR IGNORE dans les INSERT ?
#      - à quoi sert la table rejected_rows ?
#      - pourquoi charger tranches_effectifs avant entreprises ?

# TODO Q12 :
#  Approche : ETL 
# Justification : on transforme le CSV brut en 3 tables normalisées
# AVANT de charger dans SQLite. La base ne reçoit que des données
# propres et structurées, ce qui correspond exactement au pattern ETL.

# TODO Q14 :
# OR IGNORE : ???
# OR IGNORE : si on relance le pipeline, les lignes déjà insérées
#   (même PK) sont silencieusement ignorées → idempotence, pas de doublon

# rejected_rows : ???
# rejected_rows : plutôt que stopper le pipeline sur une erreur,
#   on isole les lignes problématiques ici pour les analyser après.
#   Le pipeline continue et charge le reste normalement.

# Ordre d'insertion : ???
# Ordre d'insertion : tranches_effectifs et secteurs AVANT entreprises
#   car entreprises a des FK vers ces tables. SQLite refuserait
#   d'insérer une entreprise qui pointe vers une tranche inexistante

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    handlers=[
        logging.StreamHandler(),
        logging.FileHandler('pipeline.log', encoding='utf-8')
    ]
)
log = logging.getLogger('pipeline')
DB  = 'entreprises_isere.db'

def run_pipeline():
    log.info('=== DEBUT PIPELINE ETL ===')
 
    # EXTRACT
    log.info('[EXTRACT] Chargement du fichier source...')
    df_raw = pd.read_csv(SRC)
    log.info(f'[EXTRACT] {len(df_raw)} lignes chargées')
    print(df_raw.head(2))
    # TRANSFORM
    log.info('[TRANSFORM] Communes...')
    df_c = extract_communes(SRC)
    print(df_c.head(2))
    log.info('[TRANSFORM] Secteurs...')
    df_s = extract_secteurs(SRC)
 
    log.info('[TRANSFORM] Tranches effectifs...')
    df_t = extract_tranches(SRC)
 
    log.info('[TRANSFORM] Entreprises...')
    df_e = extract_entreprises(SRC, "data/clean/communes.csv")

    log.info(f'[TRANSFORM] {len(df_c)} communes, {len(df_s)} secteurs, '
             f'{len(df_t)} tranches, {len(df_e)} entreprises')

    log.info(f'[LOAD] Chargement dans {DB}...')
    load_to_sqlite(DB, df_c, df_s, df_t, df_e)

    log.info('=== PIPELINE TERMINÉ ===')

run_pipeline()


2026-04-16 10:42:01,388 [INFO] === DEBUT PIPELINE ETL ===
2026-04-16 10:42:01,389 [INFO] [EXTRACT] Chargement du fichier source...
2026-04-16 10:42:01,393 [INFO] [EXTRACT] 490 lignes chargées
2026-04-16 10:42:01,396 [INFO] [TRANSFORM] Communes...
2026-04-16 10:42:01,404 [INFO] [TRANSFORM] Secteurs...
2026-04-16 10:42:01,409 [INFO] [TRANSFORM] Tranches effectifs...
2026-04-16 10:42:01,415 [INFO] [TRANSFORM] Entreprises...
2026-04-16 10:42:01,426 [INFO] [TRANSFORM] 49 communes, 25 secteurs, 8 tranches, 36 entreprises
2026-04-16 10:42:01,426 [INFO] [LOAD] Chargement dans entreprises_isere.db...
2026-04-16 10:42:01,473 [INFO] === PIPELINE TERMINÉ ===


  nom_commune                    nom      siren code_naf tranche_effectif  \
0    GRENOBLE               LA POSTE  356000000   53.10Z               53   
1    GRENOBLE  ELECTRICITE DE FRANCE  552081317   35.11Z               53   

                                             adresse  cp_entreprise  latitude  \
0  DIRECTION GENERALE DE LA POSTE 9 RUE DU COLONE...          75015   45.1842   
1              22-30 22 AVENUE DE WAGRAM 75008 PARIS          75008   45.1842   

   longitude                                        libelle_naf  \
0     5.7155  Activités de poste dans le cadre d'une obligat...   
1     5.7155                           Production d'électricité   

           libelle_tranche  
0  10 000 salariés et plus  
1  10 000 salariés et plus  
extract_communes : 49 communes → data/clean/communes.csv
   code_insee nom_commune  latitude  longitude
0       38001    GRENOBLE   45.1842     5.7155
10      38002      VOIRON   45.3805     5.5869
extract_secteurs : 25 → data/clean/se

In [61]:
# 🔧 TODO — Exercice 6.3 : Requêtes de validation SQLite
#
# 1. Comptez les lignes de chaque table
# 2. Vérifiez l'intégrité référentielle
# 3. Affichez le top 5 des communes par nombre d'entreprises

conn = sqlite3.connect(DB)
cur  = conn.cursor()

print('=== RAPPORT DE VALIDATION ===')

# TODO 1 : comptages
for table in ['communes', 'secteurs', 'entreprises', 'rejected_rows']:
    cur.execute(f'SELECT COUNT(*) FROM {table}')
    print(f'  {table:20s} : {cur.fetchone()[0]}')

# TODO 2 : intégrité (entreprises avec code_insee absent de communes)
cur.execute('''
    SELECT COUNT(*) FROM entreprises
    WHERE code_insee IS NOT NULL
    AND code_insee NOT IN (SELECT code_insee FROM communes)
''')
print(f'\nEntreprises avec code_insee invalide : {cur.fetchone()[0]}')

# TODO 3 : top 5 communes
cur.execute('''
    SELECT c.nom_commune, COUNT(e.siren) AS nb
    FROM communes c
    LEFT JOIN entreprises e
        ON c.code_insee = e.code_insee
    GROUP BY c.nom_commune
    ORDER BY nb DESC
    LIMIT 5
''')

print('\nTop 5 communes :')
for nom, nb in cur.fetchall():
    print(f'  {nom:30s} : {nb}')

conn.close()


=== RAPPORT DE VALIDATION ===
  communes             : 49
  secteurs             : 25
  entreprises          : 36
  rejected_rows        : 0

Entreprises avec code_insee invalide : 0

Top 5 communes :
  GRENOBLE                       : 10
  SAINT-MARTIN-D'URIAGE          : 7
  VOREPPE                        : 3
  VIZILLE                        : 3
  RIVES                          : 3


In [65]:
# ✅ CODE FOURNI — Consulter les rejected_rows

conn = sqlite3.connect(DB)
df_rej = pd.read_sql('SELECT * FROM rejected_rows', conn)
conn.close()

print(f'Total lignes rejetées : {len(df_rej)}')
if len(df_rej) > 0:
    print(df_rej[['table_name','error_msg']].head(5).to_string())
else:
    print('✓ Aucune ligne rejetée')


Total lignes rejetées : 0
✓ Aucune ligne rejetée


---
## ✅ Bilan du Kit 2

| Fichier | Description |
|---|---|
| `diagramme_uml.png` + `diagramme_uml_3NF.png` | Modèles UML |
| `memo_formes_normales.md` | Analyse 1NF/2NF/3NF |
| `data/clean/communes.csv` | Table communes |
| `data/clean/secteurs.csv` | Table secteurs (avec libellés) |
| `data/clean/tranches_effectifs.csv` | Table tranches effectifs |
| `data/clean/entreprises.csv` | Table entreprises |
| `pipeline.log` | Trace d'exécution |
| `entreprises_isere.db` | Base SQLite — 5 tables |

**Félicitations — pipeline data engineer complet !**
